# 04 — SER construction and feature engineering

The target formula `SER = sphere + cylinder / 2` is mathematically standard, but its valid application depends on confirmation of the refraction convention and measurement protocol. The supplied files contain no such documentation. Accordingly, this notebook inventories candidate features and preserves explicit stop gates; it does not create a training target or assert target compatibility. IDs and SER-derived classes are not predictors.

## SER definition, target candidates, and leakage controls

### What SER represents
Spherical Equivalent Refraction (SER) is a single dioptric value summarizing the average refractive power of the two principal meridians of an eye. In the usual sphere/cylinder representation, one principal meridian has power `Sphere` and the perpendicular meridian has power `Sphere + Cylinder`. Their mean is:

**SER = Sphere + (Cylinder / 2)**

This definition only applies when the sphere and cylinder measurements use a compatible, documented prescription convention and refer to the same eye and examination. SER is continuous (in diopters); it should not be confused with a refractive category such as myopia/emmetropia/hyperopia.

### Why convention checks matter
Cylinder can be recorded in plus-cylinder or minus-cylinder notation. Those notations can describe equivalent prescriptions, but sphere and cylinder values change when transposed. Applying the formula to values without knowing the notation, refraction method, measurement conditions, and any prior transposition can produce a different SER. Check the source protocol and codebook first. Do not infer sign convention from the fact that observed cylinders are mostly negative.

### Target candidates in these files
- **`DATA.xls`, 2018 and 2014:** `spherical refraction of right eye` plus `cylindrical refraction of right eye` are component candidates for a right-eye SER. They are not a verified SER target yet.
- **`Data3.xlsx`, sheet `2014-2017`:** the right-eye sphere/cylinder pair and left-eye sphere/cylinder pair could each define a separate eye-specific candidate, after protocol verification. Keep right and left targets distinct.
- **`DataSet_main.csv`:** `SE` is a name-based candidate only. The file does not define what it measures, its laterality, units, derivation, or whether it is continuous SER. Do not treat it as SER until documented.

### Predictor leakage exclusions
When SER is calculated from a sphere/cylinder pair, exclude both components from predictors: together they determine the target exactly, so a model using them would recover the construction formula rather than predict SER from independent information. Exclude any direct SER/SE field when it is itself the outcome; exclude refractive class labels or categories derived from SER; exclude participant/record IDs; and exclude post-outcome variables that would not be available at prediction time. Axis is not part of the formula, but should not be included automatically: verify what it encodes and whether it is available at the intended prediction time.

For the provisional `DATA.xls` 2018 experiment, candidate predictors are `gender`, `height`, `weight`, `axial length of right eye`, `time of study at weekdays`, `time of study at weekend days`, `time outdoors at weekdays`, and `time outdoors at weekend days`. The exact observed columns are listed in the readiness notebook. These names still need definition, units, coding, and timing documentation. Axial length is a candidate clinical predictor only if its measurement timing is appropriate for the intended use. No predictor set is approved until these checks pass.

In [1]:
from pathlib import Path
import shutil,subprocess,tempfile
import numpy as np,pandas as pd
ROOT=Path.cwd()
if not (ROOT/"DATA.xls").exists(): ROOT=ROOT.parent
try: xls=pd.read_excel(ROOT/"DATA.xls",sheet_name=None)
except ImportError:
 exe=shutil.which("soffice") or "/Users/omvdangi/.cache/codex-runtimes/codex-primary-runtime/dependencies/bin/override/soffice"
 tmp=Path(tempfile.mkdtemp(prefix="myopia_nb_")); subprocess.run([exe,"--headless","--convert-to","xlsx","--outdir",str(tmp),str(ROOT/"DATA.xls")],check=True,capture_output=True); xls=pd.read_excel(tmp/"DATA.xlsx",sheet_name=None)
d3=pd.read_excel(ROOT/"Data3.xlsx",sheet_name=None)
csv=pd.read_csv(ROOT/"DataSet_main.csv",sep=";",decimal=",",engine="python")
datasets={"DATA.xls/2018":xls["2018"],"DATA.xls/2014":xls["2014"],"Data3.xlsx/2014-2017":d3["2014-2017"],"DataSet_main.csv":csv}


In [2]:
mapping=[]
for cohort,df in datasets.items():
    for col in df.columns:
        label=str(col).lower()
        if any(t in label for t in ["gender","sex"]): group="demographic"
        elif any(t in label for t in ["age","birth"]): group="demographic"
        elif any(t in label for t in ["height","weight"]): group="biological"
        elif any(t in label for t in ["sphere","cyl","axial","\bal "]): group="clinical"
        elif any(t in label for t in ["study","outdoor","sleep","read","screen","activity"]): group="environmental_lifestyle"
        elif label in ["id"] or "participant" in label: group="identifier_exclude"
        else: group="unclassified_requires_codebook"
        mapping.append({"cohort":cohort,"source_column":str(col),"candidate_group":group,"standardized_name":"","definition_verified":False})
feature_map=pd.DataFrame(mapping); feature_map.to_csv(ROOT/"datasets"/"processed"/"feature_mapping.csv",index=False); display(feature_map)

# Protocol confirmation must be documented before setting True. Do not infer it from column labels.
REFRACTION_CONVENTION_VERIFIED=False
if not REFRACTION_CONVENTION_VERIFIED:
    print("STOP: no SER values created; confirm refraction convention, laterality, and protocol from source documentation first.")


,cohort,source_column,candidate_group,standardized_name,definition_verified
0,DATA.xls/2018,gender,demographic,,False
1,DATA.xls/2018,time of study at weekdays,environmental_lifestyle,,False
2,DATA.xls/2018,time of study at weekend days,environmental_lifestyle,,False
3,DATA.xls/2018,time outdoors at weekdays,environmental_lifestyle,,False
4,DATA.xls/2018,time outdoors at weekend days,environmental_lifestyle,,False
5,DATA.xls/2018,height,biological,,False
6,DATA.xls/2018,weight,biological,,False
7,DATA.xls/2018,spherical refraction of right eye,unclassified_requires_codebook,,False
8,DATA.xls/2018,cylindrical refraction of right eye,clinical,,False
9,DATA.xls/2018,axis of right eye,unclassified_requires_codebook,,False


STOP: no SER values created; confirm refraction convention, laterality, and protocol from source documentation first.


## Feature grouping and target status
`DATA.xls` includes the requested eight candidate predictors by name, but the `gender` encoding and time units are undocumented. Sphere and cylinder are outcome components and must not be used as predictors when constructing SER from them. `Data3.xlsx` has bilateral sphere/cylinder candidates and candidate axial-length fields. `DataSet_main.csv` contains a column called `SE`; no numeric transformation can establish that it means continuous spherical equivalent refraction. Age derivation from birth/exam years is approximate and should not be interpreted as exact age without dates and convention documentation.